# 🏥 Chronic Kidney Disease (CKD) - Exploratory Data Analysis & Modeling

This notebook provides a comprehensive exploration of the CKD dataset and demonstrates the machine learning pipeline for predicting:
1. **CKD Classification** (Yes/No)
2. **Kidney Function Score** (Regression)

---

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').absolute().parent / 'src'))

# Set display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Setup complete!")

## 2. Load and Explore Data

In [ ]:
# Load dataset
data_path = Path('.').absolute().parent / 'data' / 'Chronic_Kidney_Disease_Risk_Assessment.csv'
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# First few rows
df.head(10)

In [ ]:
# Data types and info
df.info()

In [ ]:
# Statistical summary
df.describe(include='all')

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).sort_values('Missing Count', ascending=False)

## 3. Target Variable Analysis

In [ ]:
# CKD distribution (Classification Target)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
ckd_counts = df['ckd'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(ckd_counts.values, labels=ckd_counts.index, autopct='%1.1f%%',
           colors=colors, explode=(0.05, 0.05), shadow=True)
axes[0].set_title('CKD Distribution', fontsize=14, fontweight='bold')

# Kidney function score histogram (Regression Target)
axes[1].hist(df['kidney_function_score'], bins=30, color='#3498db', edgecolor='white', alpha=0.7)
axes[1].axvline(df['kidney_function_score'].mean(), color='#e74c3c', linestyle='--', linewidth=2, 
               label=f'Mean: {df["kidney_function_score"].mean():.1f}')
axes[1].axvline(df['kidney_function_score'].median(), color='#2ecc71', linestyle='--', linewidth=2,
               label=f'Median: {df["kidney_function_score"].median():.1f}')
axes[1].set_xlabel('Kidney Function Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Kidney Function Score Distribution', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nCKD Distribution:\n{df['ckd'].value_counts()}")
print(f"\nKidney Function Score Stats:\n{df['kidney_function_score'].describe()}")

## 4. Numerical Features Analysis

In [ ]:
# Numerical features distribution
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [c for c in numerical_cols if c not in ['patient_id', 'kidney_function_score']]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    df[col].hist(ax=ax, bins=25, color='#3498db', edgecolor='white', alpha=0.7)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.axvline(df[col].mean(), color='#e74c3c', linestyle='--', linewidth=1.5)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots by CKD status
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    df.boxplot(column=col, by='ckd', ax=ax)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_xlabel('CKD')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features by CKD Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Categorical Features Analysis

In [ ]:
# Categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'ckd']

print(f"Categorical columns: {categorical_cols}")

In [ ]:
# CKD rate by categorical features
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ax = axes[i]
    ct = pd.crosstab(df[col], df['ckd'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], width=0.8)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Percentage')
    ax.legend(title='CKD', loc='upper right')
    ax.tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('CKD Rate by Categorical Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
numerical_data = df.select_dtypes(include=[np.number]).drop(columns=['patient_id'])
corr_matrix = numerical_data.corr()

fig, ax = plt.subplots(figsize=(14, 12))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
           cmap='RdBu_r', center=0, square=True,
           linewidths=0.5, cbar_kws={'shrink': 0.8},
           ax=ax, annot_kws={'size': 8})

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Risk Factor Analysis

In [ ]:
# Key risk factors analysis
risk_factors = ['hypertension', 'diabetes_mellitus', 'smoking', 'family_history_ckd']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, factor in enumerate(risk_factors):
    ct = pd.crosstab(df[factor], df['ckd'], normalize='index') * 100
    ckd_rates = ct['Yes'] if 'Yes' in ct.columns else pd.Series([0] * len(ct), index=ct.index)
    
    bars = axes[i].bar(ckd_rates.index, ckd_rates.values, 
                      color=['#2ecc71', '#e74c3c'] if len(ckd_rates) == 2 else ['#3498db'])
    axes[i].set_title(factor.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_ylabel('CKD Rate (%)')
    axes[i].set_ylim(0, 100)
    
    for bar in bars:
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{height:.1f}%', ha='center', fontsize=10)

plt.suptitle('CKD Rate by Key Risk Factors', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Data Preprocessing

In [ ]:
# Import our data pipeline
from data_pipeline import CKDDataPipeline

# Initialize pipeline
pipeline = CKDDataPipeline(str(data_path))
pipeline.load_data()

# Get data info
info = pipeline.get_data_info()
print(f"Dataset Shape: {info['shape']}")

In [ ]:
# Prepare classification data
X_train_cls, X_test_cls, y_train_cls, y_test_cls, feature_names = pipeline.get_classification_data()

print(f"Classification Data:")
print(f"  Training: {X_train_cls.shape}")
print(f"  Test: {X_test_cls.shape}")
print(f"  Features: {feature_names}")

In [ ]:
# Prepare regression data
pipeline_reg = CKDDataPipeline(str(data_path))
pipeline_reg.load_data()
X_train_reg, X_test_reg, y_train_reg, y_test_reg, _ = pipeline_reg.get_regression_data()

print(f"Regression Data:")
print(f"  Training: {X_train_reg.shape}")
print(f"  Test: {X_test_reg.shape}")

## 9. Classification Models

In [ ]:
from classification_models import CKDClassificationModels

# Initialize and train classification models
classifier = CKDClassificationModels()
classifier.train_all_models(X_train_cls, y_train_cls, feature_names=feature_names)

print("Classification models trained!")

In [ ]:
# Evaluate classification models
classifier.evaluate_all_models(X_test_cls, y_test_cls)

# Get comparison table
cls_comparison = classifier.get_comparison_table()
cls_comparison

In [ ]:
# Get best classifier
best_cls_name, best_cls_model = classifier.get_best_model('roc_auc')
print(f"Best Classification Model: {best_cls_name.replace('_', ' ').title()}")

## 10. Regression Models

In [ ]:
from regression_models import CKDRegressionModels

# Initialize and train regression models
regressor = CKDRegressionModels()
regressor.train_all_models(X_train_reg, y_train_reg, feature_names=feature_names)

print("Regression models trained!")

In [ ]:
# Evaluate regression models
regressor.evaluate_all_models(X_test_reg, y_test_reg)

# Get comparison table
reg_comparison = regressor.get_comparison_table()
reg_comparison

In [ ]:
# Get best regressor
best_reg_name, best_reg_model = regressor.get_best_model('r2')
print(f"Best Regression Model: {best_reg_name.replace('_', ' ').title()}")

## 11. Model Evaluation Visualizations

In [ ]:
from model_evaluation import ModelEvaluator, create_summary_dashboard

# Create evaluator
evaluator = ModelEvaluator()

# ROC Curves
fig = evaluator.plot_roc_curves(classifier.results, y_test_cls)
plt.show()

In [ ]:
# Classification comparison
fig = evaluator.plot_classification_comparison(cls_comparison)
plt.show()

In [ ]:
# Regression comparison
fig = evaluator.plot_regression_comparison(reg_comparison)
plt.show()

In [ ]:
# Summary dashboard
dashboard = create_summary_dashboard(cls_comparison, reg_comparison)
plt.show()

## 12. Feature Importance

In [ ]:
# Get feature importance from Random Forest
importance_df = classifier.get_feature_importance('random_forest')

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

top_features = importance_df.head(15)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top_features)))

bars = ax.barh(range(len(top_features)), top_features['importance'], color=colors, alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].apply(lambda x: x.replace('_', ' ').title()))
ax.invert_yaxis()

ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 13. AI Assistant Demo

In [ ]:
from ai_assistant import CKDAssistant

assistant = CKDAssistant()

# Get risk interpretation for different probabilities
for prob in [0.15, 0.45, 0.85]:
    interpretation = assistant.interpret_risk_level(prob)
    print(f"\n{'='*50}")
    print(f"Probability: {prob:.0%}")
    print(f"Risk Level: {interpretation['level']}")
    print(f"Description: {interpretation['description'][:100]}...")

In [ ]:
# Generate sample patient report
sample_patient = {
    'age': 58,
    'gender': 'Male',
    'blood_pressure_systolic': 145,
    'blood_pressure_diastolic': 92,
    'blood_glucose': 165,
    'serum_creatinine': 1.8,
    'blood_urea_nitrogen': 28,
    'hemoglobin': 11.5,
    'bmi': 28.5,
    'hypertension': 'Yes',
    'diabetes_mellitus': 'Yes',
    'smoking': 'No'
}

report = assistant.generate_patient_report(
    sample_patient, 
    ckd_probability=0.65, 
    kidney_function_score=52.5
)

print(report)

## 14. Conclusions

### Key Findings:

1. **Dataset Insights:**
   - Well-balanced target variable distribution
   - Strong correlations between kidney function score and CKD status
   - Key risk factors: age, blood pressure, creatinine, hypertension, diabetes

2. **Model Performance:**
   - Tree-based models (Random Forest, XGBoost) perform best
   - Neural networks achieve competitive results
   - Classification metrics: >90% accuracy, >95% ROC-AUC
   - Regression metrics: R² > 0.90

3. **Important Features:**
   - Serum creatinine is the most important predictor
   - Blood pressure, age, and hemoglobin are key factors
   - Medical history (hypertension, diabetes) significantly impacts predictions

### Next Steps:
- Run `python src/train.py` to train and save models
- Launch `streamlit run app.py` for the web interface
- Explore hyperparameter tuning with `--tune` flag